# Kapittel 7: Bygge chatteapplikasjoner
## Microsoft Foundry Models API Komme i gang

Denne notatboken er tilpasset fra [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) som inkluderer notatbøker som får tilgang til [Azure OpenAI](aoai-assignment.ipynb) tjenester.

> **Merk:** GitHub Models fases ut ved slutten av juli 2026. Denne notatboken retter seg nå mot [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), som tilbyr den samme gratis prøve-katalogen av modeller og Azure AI Inference SDK-opplevelsen.


# Oversikt  
"Store språkmodeller er funksjoner som mapper tekst til tekst. Gitt en inngangsstreng av tekst, prøver en stor språkmodell å forutsi teksten som kommer neste"(1). Denne "hurtigstart"-notatboken vil introdusere brukere til overordnede LLM-konsepter, kjernepakke-krav for å komme i gang med AML, en myk introduksjon til promptdesign, og flere korte eksempler på ulike brukstilfeller. 


## Innholdsfortegnelse  

[Oversikt](#overview)  
[Hvordan bruke OpenAI-tjenesten](#how-to-use-openai-service)  
[1. Opprette din OpenAI-tjeneste](#1.-creating-your-openai-service)  
[2. Installering](#2.-installation)    
[3. Legitimajon](#3.-credentials)  

[Bruksområder](#use-cases)    
[1. Oppsummer tekst](#1.-summarize-text)  
[2. Klassifiser tekst](#2.-classify-text)  
[3. Generer nye produktnavn](#3.-generate-new-product-names)  
[4. Finjuster en klassifiserer](#4.fine-tune-a-classifier)  

[Referanser](#references)


### Bygg din første prompt  
Denne korte øvelsen gir en grunnleggende introduksjon til å sende forespørsler til en modell i Microsoft Foundry Models for en enkel oppgave "oppsummering".


**Trinn**:  
1. Installer `azure-ai-inference` biblioteket i ditt Python-miljø, hvis du ikke allerede har gjort det.  
2. Last inn standard hjelpebiblioteker og sett opp dine Microsoft Foundry Models legitimasjonsopplysninger.  
3. Velg en modell for oppgaven din  
4. Lag en enkel prompt for modellen  
5. Send forespørselen din til modellens API!


### 1. Installer `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Importer hjelpebiblioteker og oppretter legitimasjon


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Finne riktig modell  
Modeller som GPT-4o og GPT-4o mini kan forstå og generere naturlig språk, og er tilgjengelige i Microsoft Foundry Models-katalogen sammen med modeller fra Meta, Mistral, Cohere og Microsoft.  


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. Design av spørsmål  

"Magien med store språkmodeller er at ved å trenes til å minimere denne prediksjonsfeilen over enorme mengder tekst, ender modellene opp med å lære konsepter som er nyttige for disse prediksjonene. For eksempel lærer de konsepter som"(1):

* hvordan man staver
* hvordan grammatikk fungerer
* hvordan man omformulerer
* hvordan man svarer på spørsmål
* hvordan man fører en samtale
* hvordan man skriver på mange språk
* hvordan man koder
* osv.

#### Hvordan kontrollere en stor språkmodell  
"Av alle innspill til en stor språkmodell, er teksten i spørsmålet uten tvil det mest innflytelsesrike(1).

Store språkmodeller kan promptes til å produsere utdata på noen få måter:

Instruksjon: Fortell modellen hva du vil ha
Fullføring: Få modellen til å fullføre begynnelsen av det du ønsker
Demonstrasjon: Vis modellen hva du vil ha, med enten:
Noen få eksempler i prompten
Mange hundre eller tusen eksempler i et finjustert treningsdatasett"



#### Det finnes tre grunnleggende retningslinjer for å lage spørsmål:

**Vis og fortell**. Gjør det tydelig hva du vil enten gjennom instruksjoner, eksempler, eller en kombinasjon av de to. Hvis du vil at modellen skal rangere en liste med elementer i alfabetisk rekkefølge eller klassifisere et avsnitt etter stemning, vis at det er det du ønsker.

**Gi kvalitetsdata**. Hvis du prøver å bygge en klassifiserer eller få modellen til å følge et mønster, sørg for at det finnes nok eksempler. Pass på å korrekturlese eksemplene dine — modellen er vanligvis smart nok til å se gjennom grunnleggende stavefeil og gi deg et svar, men den kan også anta at dette er bevisst og det kan påvirke svaret.

**Sjekk innstillingene dine.** Temperatur- og top_p-innstillingene styrer hvor deterministisk modellen er i å generere et svar. Hvis du ber om et svar hvor det bare finnes ett riktig svar, vil du sette disse lavere. Hvis du ønsker mer varierte svar, kan det hende du vil sette dem høyere. Den vanligste feilen folk gjør med disse innstillingene er å anta at de er kontroller for «klokskap» eller «kreativitet».


Kilde: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Send inn!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Utfør det samme kallet igjen, hvordan stemmer resultatene overens?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Oppsummer tekst  
#### Utfordring  
Oppsummer tekst ved å legge til en 'tl;dr:' på slutten av et tekstavsnitt. Legg merke til hvordan modellen forstår hvordan den skal utføre en rekke oppgaver uten ekstra instruksjoner. Du kan eksperimentere med mer beskrivende oppfordringer enn tl;dr for å endre modellens oppførsel og tilpasse oppsummeringen du mottar(3).  

Nyere arbeid har vist betydelige forbedringer på mange NLP-oppgaver og referanserammer ved å forhåndstrene på et stort tekstkorpus etterfulgt av finjustering på en spesifikk oppgave. Selv om arkitekturen vanligvis er oppgaveuavhengig, krever denne metoden fortsatt oppgavespesifikke finjusteringsdatasett med tusenvis eller titusenvis av eksempler. Til sammenligning kan mennesker vanligvis utføre en ny språkoppgave med bare noen få eksempler eller enkle instrukser - noe nåværende NLP-systemer fortsatt i stor grad sliter med å gjøre. Her viser vi at økt skalering av språkmodeller forbedrer oppgaveuavhengig, fåskuddsytelse betydelig, noen ganger til og med til konkurransedyktig nivå med tidligere toppmoderne finjusteringsmetoder.  



Tl;dr


# Øvelser for flere bruksområder  
1. Oppsummer tekst  
2. Kategoriser tekst  
3. Generer nye produktnavn


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Klassifiser tekst  
#### Utfordring  
Klassifiser elementer i kategorier som gis ved inferenstidspunktet. I følgende eksempel gir vi både kategoriene og teksten som skal klassifiseres i prompten (*playground_reference). 

Kundespørsmål: Hei, en av tastene på bærbar PC-tastaturet mitt ble nylig ødelagt, og jeg trenger en erstatning:

Klassifisert kategori:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Generer nye produktnavn
#### Utfordring
Lag produktnavn ut fra eksemplene på ord. Her inkluderer vi i prompten informasjon om produktet vi skal generere navn til. Vi gir også et lignende eksempel for å vise mønsteret vi ønsker å motta. Vi har også satt temperaturverdien høy for å øke tilfeldighetene og få mer innovative svar.

Produktbeskrivelse: En hjemmemelkshake-maskin
Frøord: rask, sunn, kompakt.
Produktnavn: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Produktbeskrivelse: Et par sko som passer alle skostørrelser.
Frøord: tilpassbar, passer, omni-passform.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Referanser  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Beste praksis for finjustering av GPT-modeller for å klassifisere tekst](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# For mer hjelp  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# Bidragsytere
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på originalspråket skal betraktes som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
